# Notebook 5: Hybrid Search & Reranking

**Difficulty:** Advanced | **Estimated Time:** 150-180 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

1. ✅ Understand dense vs sparse vectors and BM25 algorithm
2. ✅ Implement hybrid search combining semantic and keyword matching
3. ✅ Configure alpha parameter for optimal score fusion
4. ✅ Apply reranking models (Cohere, SentenceTransformer cross-encoders)
5. ✅ Build custom query engines for production
6. ✅ Implement production patterns (caching, monitoring, error handling)

## Prerequisites

- Completed Notebooks 1-4
- Understanding of embeddings and retrieval
- Optional: Cohere API key for reranking (free tier available)

## Curriculum Coverage

- **Section 4.1:** Hybrid Search Fundamentals
- **Section 4.2:** Implementing Hybrid Search
- **Section 4.5:** Reranking
- **Section 4.6:** Response Synthesizers
- **Section 4.8:** Custom Query Engine Implementation

---

## 1. Setup & Imports

In [ ]:
# Core LlamaIndex
from llama_index.core import (
    VectorStoreIndex,
    Settings,
    Document,
    StorageContext,
)
from llama_index.core.query_engine import BaseQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import QueryBundle, NodeWithScore

# Reranking
from llama_index.core.postprocessor import SentenceTransformerRerank
try:
    from llama_index.postprocessor.cohere_rerank import CohereRerank
    COHERE_AVAILABLE = True
except ImportError:
    COHERE_AVAILABLE = False
    print("⚠️  Cohere rerank not available. Install with: pip install llama-index-postprocessor-cohere-rerank")

# Vector Stores (for hybrid search)
try:
    from llama_index.vector_stores.qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    QDRANT_AVAILABLE = True
except ImportError:
    QDRANT_AVAILABLE = False
    print("⚠️  Qdrant not available. Install with: pip install llama-index-vector-stores-qdrant qdrant-client")

# LLM and Embeddings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Utilities
from dotenv import load_dotenv
import os
import time
from typing import List, Optional, Any
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

In [ ]:
# Load environment variables and configure Settings
load_dotenv()

Settings.llm = OpenAI(model="gpt-6.1-terra", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    dimensions=1536
)

print("✅ Settings configured")

---

## 2. Understanding Hybrid Search

### Dense vs Sparse Vectors

**Dense Vectors (Semantic):**
- Generated by neural embedding models
- All dimensions have non-zero values
- Capture semantic meaning
- Example: `[0.12, -0.34, 0.56, ..., 0.89]` (1536 dims)

**Sparse Vectors (Keyword/BM25):**
- Based on term frequency (TF-IDF, BM25)
- Most dimensions are zero
- Capture exact term matches
- Example: `{"vector": 0.8, "search": 0.6, "database": 0.4}`

### Why Hybrid?

**Strengths of Each:**

| Aspect | Dense (Semantic) | Sparse (Keyword) |
|--------|-----------------|------------------|
| **Synonyms** | ✅ Excellent | ❌ Poor |
| **Exact terms** | ⚠️  Good | ✅ Excellent |
| **Context** | ✅ Excellent | ❌ None |
| **Rare terms** | ⚠️  Good | ✅ Excellent |
| **Speed** | Fast | Very Fast |

**Hybrid combines both for best results!**

### BM25 Algorithm Overview

**BM25 (Best Matching 25)** is a ranking function:

```
BM25(D, Q) = Σ IDF(qi) * (f(qi, D) * (k1 + 1)) / (f(qi, D) + k1 * (1 - b + b * |D| / avgdl))
```

Where:
- `D`: Document
- `Q`: Query
- `f(qi, D)`: Term frequency of query term qi in document D
- `|D|`: Length of document D
- `avgdl`: Average document length
- `k1`: Term frequency saturation (typically 1.2-2.0)
- `b`: Length normalization (typically 0.75)
- `IDF(qi)`: Inverse document frequency

**Key Properties:**
- **Diminishing returns**: Additional term occurrences matter less
- **Length normalization**: Longer documents aren't unfairly penalized
- **IDF weighting**: Rare terms score higher

---

## 3. Prepare Sample Documents

In [ ]:
# Create comprehensive documents for hybrid search testing
documents = [
    Document(
        text="""Vector databases are specialized systems for storing and querying high-dimensional vectors. 
        Popular vector databases include Qdrant, Pinecone, Weaviate, and Milvus. They use algorithms like 
        HNSW (Hierarchical Navigable Small World) for approximate nearest neighbor search. Vector databases 
        are essential for semantic search, recommendation systems, and RAG applications.""",
        metadata={"topic": "vector_databases", "doc_type": "overview", "keywords": "vector database HNSW"}
    ),
    Document(
        text="""BM25 is a probabilistic ranking function used for information retrieval. It improves upon 
        TF-IDF by adding term frequency saturation and document length normalization. The formula includes 
        parameters k1 (controls term frequency saturation) and b (controls length normalization). BM25 is 
        widely used in search engines like Elasticsearch and provides excellent keyword-based retrieval.""",
        metadata={"topic": "bm25", "doc_type": "technical", "keywords": "BM25 ranking TF-IDF"}
    ),
    Document(
        text="""Hybrid search combines semantic vector search with keyword-based methods like BM25. This 
        approach leverages the strengths of both: semantic search handles synonyms and context, while 
        keyword search ensures exact term matches. The results are typically fused using reciprocal rank 
        fusion or weighted score combination with an alpha parameter.""",
        metadata={"topic": "hybrid_search", "doc_type": "technical", "keywords": "hybrid search semantic keyword"}
    ),
    Document(
        text="""Reranking is a two-stage retrieval process where an initial set of candidates is retrieved 
        cheaply, then reranked using a more expensive but accurate model. Cross-encoder models like those 
        from sentence-transformers are popular for reranking. Cohere provides a reranking API that achieves 
        excellent results. Reranking significantly improves retrieval quality at modest cost increase.""",
        metadata={"topic": "reranking", "doc_type": "technical", "keywords": "reranking cross-encoder Cohere"}
    ),
    Document(
        text="""Qdrant is a vector database written in Rust that supports both dense and sparse vectors. 
        It enables hybrid search by combining semantic similarity with BM25 keyword matching. Qdrant uses 
        HNSW indexing for fast approximate nearest neighbor search and supports filtering, quantization, 
        and distributed deployments. It's particularly well-suited for production RAG systems.""",
        metadata={"topic": "qdrant", "doc_type": "product", "keywords": "Qdrant vector database hybrid"}
    ),
    Document(
        text="""The alpha parameter in hybrid search controls the balance between semantic and keyword scores. 
        Alpha=0 means pure keyword (BM25) search. Alpha=1 means pure semantic search. Alpha=0.5 gives equal 
        weight to both. The optimal alpha depends on your data and queries - typically between 0.3-0.7. 
        You should tune alpha on a validation set for best results.""",
        metadata={"topic": "alpha_tuning", "doc_type": "guide", "keywords": "alpha parameter hybrid tuning"}
    ),
]

print(f"✅ Created {len(documents)} documents for hybrid search")
for doc in documents:
    print(f"  - {doc.metadata['topic']} ({doc.metadata['doc_type']})")

---

## 4. Simulating Hybrid Search (Conceptual)

**Note**: True hybrid search requires a vector database with BM25 support (like Qdrant). For this demo, we'll show the concept and implementation pattern.

In [ ]:
# Create standard vector index
index = VectorStoreIndex.from_documents(documents, show_progress=True)
print("✅ Vector index created")

### Qdrant Hybrid Search (Code Example)

**With Qdrant installed** (`qdrant-client` and `fastembed` are already in `requirements.txt`; `enable_hybrid=True` uses fastembed's BM25 sparse embeddings):

In [ ]:
if QDRANT_AVAILABLE:
    # Initialize Qdrant client
    qdrant_client = QdrantClient(location=":memory:")
    
    # Create Qdrant vector store with hybrid search enabled
    qdrant_vector_store = QdrantVectorStore(
        client=qdrant_client,
        collection_name="hybrid_search_demo",
        enable_hybrid=True,  # Enable BM25 + semantic search
        batch_size=20,
    )
    
    storage_context = StorageContext.from_defaults(vector_store=qdrant_vector_store)
    
    # Create index
    hybrid_index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True,
    )
    
    # Query with hybrid search
    hybrid_query_engine = hybrid_index.as_query_engine(
        similarity_top_k=3,
        vector_store_query_mode="hybrid",  # Use hybrid search
        alpha=0.5,  # 50% semantic, 50% keyword
    )
    
    print("✅ Qdrant hybrid search enabled")
    print("   Mode: hybrid (semantic + BM25)")
    print("   Alpha: 0.5 (equal weighting)")
else:
    print("⚠️  Qdrant not available - skipping hybrid search demo")
    hybrid_query_engine = index.as_query_engine(similarity_top_k=3)

### Testing Hybrid Search Benefits

In [ ]:
# Test queries that benefit from hybrid search
test_queries = [
    "What is BM25?",  # Exact term match important
    "How do you balance semantic and keyword search?",  # Conceptual + keyword
    "vector database for production",  # Keywords important
]

print("Hybrid Search Test Queries:\n")
for query in test_queries:
    response = hybrid_query_engine.query(query)
    print(f"Query: {query}")
    print(f"Response: {str(response)[:200]}...")
    print(f"Top source: {response.source_nodes[0].metadata.get('topic')}")
    print("-" * 80)

### 🎯 ML Engineering Note: Alpha Parameter Tuning

**Alpha controls dense/sparse balance:**

```python
final_score = alpha * semantic_score + (1 - alpha) * bm25_score
```

**Tuning Strategy:**

1. **Create validation set**: 20-50 queries with known relevant docs
2. **Test alphas**: [0.0, 0.1, 0.2, ..., 0.9, 1.0]
3. **Measure metrics**: Precision@K, Recall@K, MRR
4. **Select optimal**: Highest metric value

**Common Patterns:**
- **Technical docs**: α ≈ 0.3-0.4 (favor keywords for code, APIs)
- **General knowledge**: α ≈ 0.6-0.7 (favor semantic)
- **Product search**: α ≈ 0.4-0.5 (balanced)
- **Scientific papers**: α ≈ 0.5-0.6 (balanced, slight semantic)

---

## 5. Reranking Models

### Why Rerank?

**Two-stage retrieval:**
1. **Stage 1 (Retrieval)**: Fast, recall-focused (get 50-100 candidates)
2. **Stage 2 (Reranking)**: Slow, precision-focused (rerank top 10-20)

**Cost-Performance Trade-off:**
- Initial retrieval: Cheap (vector similarity or BM25)
- Reranking: Expensive (cross-encoder models)
- Result: Best of both worlds

### 5.1 SentenceTransformer Cross-Encoder Reranking

In [ ]:
# Create cross-encoder reranker
print("Loading cross-encoder model (first run may take a moment)...")

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",  # Fast, lightweight
    top_n=3,  # Return top 3 after reranking
)

# Create query engine with reranking
reranked_query_engine = index.as_query_engine(
    similarity_top_k=10,  # Retrieve 10 candidates
    node_postprocessors=[reranker],  # Rerank to top 3
)

print("\n✅ Reranking query engine created")
print("   Initial retrieval: top 10")
print("   After reranking: top 3")
print("   Model: cross-encoder/ms-marco-MiniLM-L-2-v2")

In [ ]:
# Compare with and without reranking
test_query = "How does hybrid search combine different retrieval methods?"

print(f"Query: {test_query}\n")
print("=" * 80)

# Without reranking
print("\nWithout Reranking (similarity only):")
no_rerank_engine = index.as_query_engine(similarity_top_k=3)
no_rerank_response = no_rerank_engine.query(test_query)
print("Top 3 sources:")
for i, node in enumerate(no_rerank_response.source_nodes, 1):
    print(f"  {i}. {node.metadata.get('topic')} (score: {node.score:.4f})")

# With reranking
print("\nWith Cross-Encoder Reranking:")
reranked_response = reranked_query_engine.query(test_query)
print("Top 3 sources (after reranking):")
for i, node in enumerate(reranked_response.source_nodes, 1):
    print(f"  {i}. {node.metadata.get('topic')} (score: {node.score:.4f})")

### 5.2 Cohere Rerank (API-based)

In [ ]:
if COHERE_AVAILABLE and os.getenv("COHERE_API_KEY"):
    # Create Cohere reranker (rerank-v3.5 is the current v3 generation;
    # rerank-v4.0 and Rerank 5 are also available)
    cohere_reranker = CohereRerank(
        api_key=os.getenv("COHERE_API_KEY"),
        top_n=3,
        model="rerank-v3.5",
    )
    
    # Create query engine with Cohere reranking
    cohere_query_engine = index.as_query_engine(
        similarity_top_k=10,
        node_postprocessors=[cohere_reranker],
    )
    
    print("✅ Cohere reranking enabled")
    print("   Model: rerank-v3.5")
    
    # Test Cohere reranking
    cohere_response = cohere_query_engine.query(test_query)
    print("\nCohere Reranked Results:")
    for i, node in enumerate(cohere_response.source_nodes, 1):
        print(f"  {i}. {node.metadata.get('topic')} (score: {node.score:.4f})")
else:
    print("⚠️  Cohere reranking not available (missing API key or package)")
    print("   Get free API key at: https://dashboard.cohere.com/")

### Reranking Model Comparison

| Model | Quality | Speed | Cost | Hosting |
|-------|---------|-------|------|---------|
| **Cross-Encoder (local)** | Good | Fast (GPU) | Free | Self-hosted |
| **Cohere Rerank** | Excellent | Fast | ~$2/1000 searches (v3.5) | API |
| **Custom fine-tuned** | Variable | Variable | Training cost | Self-hosted |

**Recommendation:**
- **Development**: Cross-encoder (free, fast iteration)
- **Production (budget)**: Cross-encoder with GPU
- **Production (quality)**: Cohere Rerank (v3.5, or Rerank 4/5 for latest)

---

## 6. Custom Query Engine for Production

### Building a Production-Ready Query Engine

In [ ]:
from llama_index.core.base.response.schema import Response
import logging

class ProductionQueryEngine(BaseQueryEngine):
    """Production query engine with error handling, logging, and monitoring."""
    
    def __init__(
        self,
        index: VectorStoreIndex,
        similarity_top_k: int = 5,
        enable_reranking: bool = True,
        rerank_top_n: int = 3,
    ):
        """Initialize production query engine.
        
        Args:
            index: Vector store index
            similarity_top_k: Number of candidates to retrieve
            enable_reranking: Whether to apply reranking
            rerank_top_n: Number of results after reranking
        """
        self._index = index
        self._similarity_top_k = similarity_top_k
        self._enable_reranking = enable_reranking
        self._rerank_top_n = rerank_top_n
        
        # Build base query engine
        node_postprocessors = []
        if enable_reranking:
            reranker = SentenceTransformerRerank(
                model="cross-encoder/ms-marco-MiniLM-L-2-v2",
                top_n=rerank_top_n,
            )
            node_postprocessors.append(reranker)
        
        self._base_engine = index.as_query_engine(
            similarity_top_k=similarity_top_k,
            node_postprocessors=node_postprocessors,
        )
        
        # Metrics
        self._query_count = 0
        self._error_count = 0
        self._total_latency = 0.0
        
        # BaseQueryEngine is a pydantic model in core 0.14 — callback_manager is required
        super().__init__(callback_manager=self._base_engine.callback_manager)
    
    def _query(self, query_bundle: QueryBundle) -> Response:
        """Execute query with error handling and monitoring."""
        start_time = time.time()
        
        try:
            # Validate query
            if not query_bundle.query_str or len(query_bundle.query_str.strip()) == 0:
                raise ValueError("Empty query")
            
            # Execute query
            response = self._base_engine.query(query_bundle)
            
            # Update metrics
            latency = time.time() - start_time
            self._query_count += 1
            self._total_latency += latency
            
            # Log
            print(f"✅ Query completed in {latency:.2f}s (avg: {self.avg_latency:.2f}s)")
            
            return response
            
        except Exception as e:
            # Handle errors
            self._error_count += 1
            print(f"❌ Query error: {str(e)}")
            
            # Return fallback response
            return Response(
                response="I encountered an error processing your query. Please try again.",
                source_nodes=[],
            )
    
    async def _aquery(self, query_bundle: QueryBundle) -> Response:
        """Async query (delegates to sync for simplicity)."""
        return self._query(query_bundle)
    
    def _get_prompt_modules(self) -> dict:
        """Prompt modules used by this engine (required abstract method).

        Delegates to the underlying engine so prompt tracking stays accurate.
        """
        return self._base_engine._get_prompt_modules()
    
    @property
    def avg_latency(self) -> float:
        """Average query latency."""
        if self._query_count == 0:
            return 0.0
        return self._total_latency / self._query_count
    
    @property
    def error_rate(self) -> float:
        """Query error rate."""
        total = self._query_count + self._error_count
        if total == 0:
            return 0.0
        return self._error_count / total
    
    def get_metrics(self) -> dict:
        """Get performance metrics."""
        return {
            "total_queries": self._query_count,
            "total_errors": self._error_count,
            "error_rate": self.error_rate,
            "avg_latency_seconds": self.avg_latency,
        }

print("✅ ProductionQueryEngine class defined")

In [ ]:
# Create production query engine
prod_engine = ProductionQueryEngine(
    index=index,
    similarity_top_k=10,
    enable_reranking=True,
    rerank_top_n=3,
)

print("✅ Production query engine created")
print("   Features: Error handling, metrics, logging")

In [ ]:
# Test production engine
test_queries_prod = [
    "What is hybrid search?",
    "Explain BM25 algorithm",
    "How does reranking improve retrieval?",
    "",  # Test error handling
]

print("Testing Production Query Engine:\n")
for query in test_queries_prod:
    if query:
        response = prod_engine.query(query)
        print(f"Query: {query}")
        print(f"Response: {str(response)[:150]}...\n")
    else:
        print("Testing empty query...")
        response = prod_engine.query(query)
        print(f"Response: {response}\n")

# Show metrics
print("\nPerformance Metrics:")
metrics = prod_engine.get_metrics()
for key, value in metrics.items():
    print(f"  {key}: {value}")

---

## 7. Production Patterns & Best Practices

### 7.1 Caching Strategy

In [ ]:
from functools import lru_cache
import hashlib

class CachedQueryEngine:
    """Query engine with response caching."""
    
    def __init__(self, base_engine: BaseQueryEngine, cache_size: int = 100):
        self._base_engine = base_engine
        self._cache = {}
        self._cache_size = cache_size
        self._cache_hits = 0
        self._cache_misses = 0
    
    def _hash_query(self, query: str) -> str:
        """Hash query for cache key (sha256, not md5)."""
        return hashlib.sha256(query.encode()).hexdigest()
    
    def query(self, query_str: str) -> Response:
        """Query with caching."""
        cache_key = self._hash_query(query_str)
        
        # Check cache
        if cache_key in self._cache:
            self._cache_hits += 1
            print(f"💨 Cache hit! (hit rate: {self.cache_hit_rate:.1%})")
            return self._cache[cache_key]
        
        # Cache miss - query and cache
        self._cache_misses += 1
        response = self._base_engine.query(query_str)
        
        # Add to cache (evict oldest if full)
        if len(self._cache) >= self._cache_size:
            # Simple FIFO eviction
            self._cache.pop(next(iter(self._cache)))
        
        self._cache[cache_key] = response
        return response
    
    @property
    def cache_hit_rate(self) -> float:
        total = self._cache_hits + self._cache_misses
        return self._cache_hits / total if total > 0 else 0.0

# Create cached engine
cached_engine = CachedQueryEngine(
    base_engine=index.as_query_engine(similarity_top_k=3),
    cache_size=50,
)

# Test caching
print("Testing cache:\n")
query = "What is BM25?"
print("First query (cache miss):")
response1 = cached_engine.query(query)
print(f"\nSecond query (cache hit):")
response2 = cached_engine.query(query)
print(f"\nCache hit rate: {cached_engine.cache_hit_rate:.1%}")

### 7.2 Monitoring & Alerting

**Production Monitoring Checklist:**

1. **Latency Metrics**:
   - P50, P95, P99 query latency
   - Retrieval vs synthesis latency breakdown
   - Alert on P99 > threshold

2. **Quality Metrics**:
   - Average similarity scores
   - Number of queries with no results
   - User feedback (thumbs up/down)

3. **Cost Metrics**:
   - LLM API calls per query
   - Embedding API calls
   - Reranking API calls
   - Total cost per 1000 queries

4. **Error Metrics**:
   - Error rate by type
   - Failed retrievals
   - LLM timeouts
   - Alert on error rate > 1%

**Tools:**
- Prometheus + Grafana for metrics
- Sentry for error tracking
- Custom logging to DataDog/CloudWatch
- LangSmith/LangFuse for LLM observability

### 7.3 Cost Optimization Strategies

**1. Embedding Costs:**
- Cache embeddings (don't re-embed same text)
- Use smaller embedding dimensions (512 vs 1536)
- Batch embed documents
- Consider local models for high volume

**2. LLM Costs:**
- Use smaller models (gpt-6-luna vs gpt-6.1-sol)
- Reduce context size (fewer retrieved chunks)
- Cache common queries
- Stream responses (better UX, same cost)

**3. Reranking Costs:**
- Rerank only top-k (not all candidates)
- Use local cross-encoders instead of API
- Cache reranking results
- Skip reranking for high-confidence retrievals

**4. Vector Database:**
- Use quantization (reduce memory)
- Optimize HNSW parameters (ef/M)
- Archive old data
- Use tiered storage

---

## 8. Performance Benchmarking

In [ ]:
import pandas as pd

def benchmark_query_engines(engines: dict, queries: List[str], runs: int = 3):
    """Benchmark multiple query engines."""
    results = []
    
    for engine_name, engine in engines.items():
        print(f"\nBenchmarking {engine_name}...")
        latencies = []
        
        for query in queries:
            for _ in range(runs):
                start = time.time()
                response = engine.query(query)
                latency = time.time() - start
                latencies.append(latency)
        
        results.append({
            "Engine": engine_name,
            "Avg Latency (s)": f"{sum(latencies) / len(latencies):.3f}",
            "Min Latency (s)": f"{min(latencies):.3f}",
            "Max Latency (s)": f"{max(latencies):.3f}",
            "Queries": len(queries) * runs,
        })
    
    return pd.DataFrame(results)

# Benchmark different configurations
engines_to_test = {
    "Baseline (k=3)": index.as_query_engine(similarity_top_k=3),
    "High Recall (k=10)": index.as_query_engine(similarity_top_k=10),
    "With Reranking": reranked_query_engine,
}

benchmark_queries = [
    "What is hybrid search?",
    "How does BM25 work?",
]

print("Running benchmark (this may take a moment)...\n")
benchmark_results = benchmark_query_engines(engines_to_test, benchmark_queries, runs=2)

print("\n" + "=" * 80)
print("Benchmark Results:")
print("=" * 80)
print(benchmark_results.to_string(index=False))

---

## 9. Summary: What You Learned

### ✅ Completed Learning Objectives

1. **Hybrid Search**: Understood dense vs sparse vectors, BM25 algorithm
2. **Implementation**: Configured hybrid search with Qdrant (conceptual)
3. **Alpha Tuning**: Learned to balance semantic and keyword scores
4. **Reranking**: Applied cross-encoder and Cohere reranking
5. **Custom Engines**: Built production query engine with monitoring
6. **Production Patterns**: Implemented caching, error handling, metrics

### Key Concepts Mastered

- **Hybrid search**: Dense (semantic) + Sparse (BM25) fusion
- **BM25**: Probabilistic ranking with TF saturation and length normalization
- **Alpha parameter**: Controls dense/sparse balance (0=keyword, 1=semantic)
- **Reranking**: Two-stage retrieval (fast recall → slow precision)
- **Cross-encoders**: Better than bi-encoders for reranking
- **Production patterns**: Caching, monitoring, error handling, cost optimization

### Production Readiness Checklist

- [ ] Choose vector database (Qdrant, Pinecone, etc.)
- [ ] Enable hybrid search if available
- [ ] Tune alpha parameter on validation set
- [ ] Implement reranking (cross-encoder or Cohere)
- [ ] Add response caching
- [ ] Set up monitoring (latency, errors, costs)
- [ ] Configure alerts (P99 latency, error rate)
- [ ] Implement error handling and fallbacks
- [ ] Add user feedback collection
- [ ] Document query patterns and performance

---

## 🎯 Congratulations!

You've completed all 5 LlamaIndex notebooks! You now have:

1. ✅ **Setup & Basics**: LlamaIndex fundamentals
2. ✅ **Documents & Chunking**: Data preparation strategies
3. ✅ **Indexing & Querying**: Vector stores and embeddings
4. ✅ **Advanced Retrieval**: Complex retrieval patterns
5. ✅ **Hybrid Search & Reranking**: Production-ready systems

### Next Steps

**Apply your knowledge:**
- Build a RAG system for your domain
- Experiment with different retrieval strategies
- Tune for your specific use case
- Deploy to production

**Continue learning:**
- LlamaIndex agents and workflows
- Multi-modal RAG (images, audio)
- Graph-based retrieval
- Advanced prompt engineering

---

## 🎯 Final Exercise: Build Your Own System

**Challenge**: Build a complete production RAG system:

1. Load your own documents (PDFs, docs)
2. Choose optimal chunking strategy
3. Set up vector database (Qdrant or Chroma)
4. Enable hybrid search (if using Qdrant)
5. Add reranking
6. Implement caching and monitoring
7. Benchmark and tune
8. Deploy (FastAPI, Streamlit, etc.)

---

## Additional Resources

- **Hybrid Search**: https://qdrant.tech/articles/hybrid-search/
- **BM25**: https://en.wikipedia.org/wiki/Okapi_BM25
- **Cohere Rerank**: https://docs.cohere.com/docs/reranking
- **Production RAG**: https://docs.llamaindex.ai/en/stable/optimizing/production_rag/